## 1. Imports and configuration

In [ ]:
import os
import sys
import random
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import yaml

from config import get_config
from data.loader import get_dataloaders
from models.model_vit import create_model
from models.jetbench_transfer import AdaptedMomentViT, read_trusted_jetbench_checkpoint, transfer_backbone
from train_utils.moment_training import run_moment_training_loop

# Command-line: python train_t03.py --config path.yml
# In Jupyter set T03_CONFIG environment variable before running cells.
if 'ipykernel' in sys.modules:
    CONFIG = os.environ.get('T03_CONFIG', 'experiments/exp_03_cumulative_moment_ablation/config/E2_m1_m2_pretrained.yml')
    cfg = get_config(CONFIG)
else:
    cfg = get_config()
with open(CONFIG if 'ipykernel' in sys.modules else next(
        sys.argv[i + 1] for i in range(len(sys.argv)-1) if sys.argv[i]=='--config'
    ), encoding='utf8') as f:
    raw_cfg = yaml.safe_load(f)
assert cfg.dataset_type == 'moment'
assert raw_cfg.get('transfer_mode') == 'jetbench_full', 'Only jetbench_full is supported by this entrypoint'
assert len(cfg.moments) in range(1, 6)
assert cfg.moments == list(range(1, len(cfg.moments)+1))
assert cfg.test_csv != cfg.val_csv
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
print('Experiment:', cfg.model_tag, 'moments:',cfg.moments, 'output:', cfg.output_dir)


## 2. Data loaders and device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert torch.cuda.is_available(), 'T03 benchmark requires allocated CUDA GPU'
train_loader, val_loader, test_loader = get_dataloaders(cfg, device=device)
assert (len(train_loader.dataset),len(val_loader.dataset),len(test_loader.dataset)) == (11200,1400,1400)
print('Dataset splits:',len(train_loader.dataset),len(val_loader.dataset),len(test_loader.dataset))
# The test loader is intentionally never evaluated in this notebook.


## 3. Create ViT-Tiny, adapt moments, transfer checkpoint

In [ ]:
channels = len(cfg.moments)
# Internal backbone always uses three channels, matching the JetBench checkpoint.
base, _unused_optimizer = create_model(
    backbone=cfg.backbone,
    input_shape=(3,32,32),
    learning_rate=cfg.learning_rate,
    dataset_type='moment',
)
source_path = Path(raw_cfg['checkpoint_path'])
assert source_path.is_file(), f'Checkpoint missing: {source_path}'
old_ckpt, old_state = read_trusted_jetbench_checkpoint(source_path)
report = transfer_backbone(base,old_state,min_coverage=0.99)
assert report['coverage'] >= .99
print('Checkpoint backbone transfer:', report)
model = AdaptedMomentViT(base_model=base,channels=channels).to(device)
assert all(p.requires_grad for p in model.parameters()), 'Full fine-tuning must keep all parameters trainable'
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
criterion = {'alpha_output':nn.CrossEntropyLoss(), 'q0_output':nn.CrossEntropyLoss()}
# All five experiments: independently reload the SAME JetBench checkpoint.


## 4. Scheduler and output manifest

In [ ]:
scheduler_cfg = cfg.scheduler
assert scheduler_cfg['type'] == 'ReduceLROnPlateau'
assert scheduler_cfg.get("mode", "max") in ("min", "max"), (
    f"Unsupported scheduler mode: {scheduler_cfg}"
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=scheduler_cfg.get("mode", "max"),
    factor=float(scheduler_cfg.get("factor", 0.5)),
    patience=int(scheduler_cfg.get("patience", 4)),
    min_lr=float(scheduler_cfg.get("min_lr", 0.0)),
)
out = Path(cfg.output_dir)
out.mkdir(parents=True, exist_ok=True)
# Refuse silently overwriting an earlier run of the same experiment.
if (out/'moment_epoch_metrics.csv').exists() or (out/'last_model.pth').exists():
    raise FileExistsError(f'Existing run outputs at {out}; use a fresh output directory or implement explicit resume')
(out/'experiment_config.yml').write_text(yaml.safe_dump(raw_cfg,sort_keys=False))
try:
    import subprocess
    revision = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
except Exception:
    revision = 'unknown'
(out/'run_manifest.json').write_text(json.dumps({
    'git_commit':revision, 'source_checkpoint':str(source_path),
    'source_checkpoint_epoch':int(old_ckpt.get('epoch',-1)),
    'backbone_coverage':report['coverage'], 'moments':cfg.moments,
    'seed':cfg.seed, 'device':str(device)},indent=2))
print('Outputs:', out)


## 5. Train and validate

In [ ]:
best_epoch,best_acc,best_metrics = run_moment_training_loop(
    cfg,train_loader,val_loader,device,model,criterion,optimizer,scheduler=scheduler,start_epoch=0
)
print('Best validation epoch:',best_epoch,'joint accuracy:',best_acc)
print('Best metrics:',best_metrics)


## 6. Training artifact summary

In [ ]:
assert (out/'best_model.pth').is_file()
assert (out/'last_model.pth').is_file()
assert (out/'moment_epoch_metrics.csv').is_file()
print('Saved:',out/'moment_epoch_metrics.csv')
print('Held-out test set was not evaluated.')


## 7. Restore best checkpoint and generate validation-only experiment reports

Plots and summaries are generated from the best checkpoint and full validation split. The held-out test split is never evaluated.

In [ ]:
from train_utils.t03_reporting import generate_t03_report

# Reporting is isolated from training: checkpoints are already saved before this stage.
try:
    best_report = generate_t03_report(
        model=model,
        validation_loader=val_loader,
        device=device,
        output_dir=cfg.output_dir,
    )
except Exception as exc:
    import traceback
    traceback.print_exc()
    print(f"[WARN] Reporting failed, but training artifacts remain intact: {exc}")
    print("Rerun the reporting step after fixing the issue; do not repeat training.")
